Luiz Jardel  
Ana Rebeca

PROBLEMA : SISTEMA DE GESTÃO DE CINEMA E INGRESSOS

In [ ]:

PRECOS = {"inteira": 30.0, "meia": 15.0, "vip": 45.0}   
sessoes = []                                            



def assentos_cinema(fileiras, por_fileira):
    
    letras = "ABCDEFGHIJ"
    lista_assentos = []
    for i in range(fileiras):
        for n in range(1, por_fileira + 1):
            lista_assentos.append(letras[i] + str(n))
    return lista_assentos


def gerar_assentos_por_capacidade(capacidade, por_fileira=10):
    
    letras = "ABCDEFGHIJ"
    lista_assentos = []
    for i in range(capacidade):
        fileira = letras[i // por_fileira]
        numero = i % por_fileira + 1
        lista_assentos.append(fileira + str(numero))
    return lista_assentos


def criar_sessao(id_sessao, filme, horario, sala, assentos):
    return {
        "id_sessao": id_sessao,
        "filme": filme,
        "horario": horario,
        "sala": sala,
        "assentos": assentos,
        "ingressos": [],
    }


def proximo_id(lista):
    if len(lista) == 0:
        return 1
    else:
        return lista[-1]["id_sessao"] + 1


def assentos_ocupados(sessao):
    ocupados = []
    for ingresso in sessao["ingressos"]:
        ocupados.append(ingresso["assento"])
    return ocupados


def assentos_livres(sessao):
    """Calculado na hora (assentos - ocupados), nunca guardado: não fica inconsistente."""
    ocupados = assentos_ocupados(sessao)
    livres = []
    for assento in sessao["assentos"]:
        if assento not in ocupados:
            livres.append(assento)
    return livres


def ler_inteiro(mensagem):
    
    while True:
        texto = input(mensagem).strip()
        if texto.isdigit() and int(texto) > 0:
            return int(texto)
        print("Entrada inválida. Digite um número inteiro maior que zero.")


def ler_texto(mensagem):
    
    while True:
        texto = input(mensagem).strip()
        if texto != "":
            return texto
        print("Entrada inválida. O campo não pode ficar vazio.")



def buscar_sessao(id_sessao):
    
    for s in sessoes:
        if s["id_sessao"] == id_sessao:
            return s
    return None


def cadastrar_sessao(id_sessao, filme, sala, horario, capacidade):
    
    if buscar_sessao(id_sessao) is not None:
        return False, "Já existe uma sessão com o ID " + str(id_sessao) + "."
    if filme.strip() == "" or sala.strip() == "" or horario.strip() == "":
        return False, "Filme, sala e horário não podem ficar vazios."
    if capacidade < 1 or capacidade > 100:
        return False, "Capacidade inválida (use de 1 a 100 assentos)."

    assentos = gerar_assentos_por_capacidade(capacidade)
    sessoes.append(criar_sessao(id_sessao, filme.strip(), horario.strip(), sala.strip(), assentos))
    return True, "Sessão " + str(id_sessao) + " cadastrada com " + str(capacidade) + " assentos."


def atualizar_sessao(id_sessao, campo, novo_valor):
    
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."
    if campo not in ("filme", "horario", "sala"):
        return False, "Campo inválido (use filme, horario ou sala)."
    if novo_valor.strip() == "":
        return False, "O novo valor não pode ser vazio."
    sessao[campo] = novo_valor.strip()
    return True, "Campo '" + campo + "' atualizado para: " + novo_valor.strip()


def listar_assentos(id_sessao):
    
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."
    ocupados = assentos_ocupados(sessao)
    print("\nSessão", sessao["id_sessao"], "-", sessao["filme"], "|", sessao["horario"], "|", sessao["sala"])
    print("(XX = ocupado)")
    fileira_atual = ""
    linha = ""
    for assento in sessao["assentos"]:
        if assento[0] != fileira_atual:            # mudou de fileira: imprime a anterior
            if linha != "":
                print(linha)
            fileira_atual = assento[0]
            linha = ""
        if assento in ocupados:
            linha += "XX".ljust(5)
        else:
            linha += assento.ljust(5)
    print(linha)
    livres = len(assentos_livres(sessao))
    return True, "Livres: " + str(livres) + " | Ocupados: " + str(len(ocupados)) + " | Total: " + str(len(sessao["assentos"]))


def sessao_lotada(id_sessao):
    """Função auxiliar: True se a sessão existe e não tem nenhum assento livre."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False
    return len(assentos_livres(sessao)) == 0



dados_iniciais = [
    ("Interestelar",      "Sala 1", "14:00", 12),
    ("Interestelar",      "Sala 1", "19:30", 12),
    ("Divertida Mente 2", "Sala 2", "15:00", 10),
    ("Divertida Mente 2", "Sala 2", "17:30", 10),
    ("Duna: Parte Dois",  "Sala 3", "20:00", 15),
    ("Duna: Parte Dois",  "Sala 3", "22:30", 15),
    ("Homem-Aranha",      "Sala 4", "16:00", 8),
    ("Homem-Aranha",      "Sala 4", "18:30", 8),
    ("Matrix",            "Sala 5", "21:00", 6),
    ("Coringa",           "Sala 6", "19:00", 8),
    ("Barbie",            "Sala 7", "13:30", 10),
    ("Pré-estreia",       "Sala 8", "23:00", 3),     # pequena: já nasce lotada
]
for (filme, sala, hor, cap) in dados_iniciais:
    cadastrar_sessao(proximo_id(sessoes), filme, sala, hor, cap)


ingressos_iniciais = {
    1: [("A1", "inteira"), ("A2", "meia"), ("B1", "vip")],
    2: [("A1", "inteira"), ("A2", "inteira"), ("A3", "meia"), ("B1", "vip")],
    3: [("A1", "meia"), ("A2", "meia")],
    5: [("A1", "vip"), ("A2", "vip"), ("A3", "inteira")],
    7: [("A1", "inteira"), ("A2", "meia")],
    9: [("A1", "vip"), ("A2", "inteira")],
    12: [("A1", "inteira"), ("A2", "meia"), ("A3", "vip")],  
}
for id_s in ingressos_iniciais:
    for (assento, tipo) in ingressos_iniciais[id_s]:
        buscar_sessao(id_s)["ingressos"].append({"assento": assento, "tipo": tipo, "preco": PRECOS[tipo]})

print("Total de sessões:", len(sessoes))
print("Exemplo de sessão (dicionário):")
print(sessoes[0])



total_ok = 0
total_testes = 0

def testar(numero, nome, entrada, esperado, observado):
    global total_ok, total_testes
    total_testes += 1
    passou = (esperado == observado)
    if passou:
        total_ok += 1
    print("Teste", numero, "-", nome)
    print("   Entrada :", entrada)
    print("   Esperado:", esperado)
    print("   Obtido  :", observado)
    print("   Resultado:", "PASSOU" if passou else "FALHOU")


ok, msg = cadastrar_sessao(99, "Filme Teste", "Sala T", "10:00", 6)
testar(1, "Cadastrar nova sessão", "id 99, capacidade 6",
    (True, ["A1", "A2", "A3", "A4", "A5", "A6"]), (ok, buscar_sessao(99)["assentos"]))


testar(2, "Buscar sessão existente", "id 1", "Interestelar", buscar_sessao(1)["filme"])


testar(3, "Buscar sessão inexistente", "id 1234", None, buscar_sessao(1234))


sessao_teste = buscar_sessao(99)
sessao_teste["ingressos"].append({"assento": "A1", "tipo": "inteira", "preco": 30.0})
atualizar_sessao(99, "horario", "21:45")
testar(4, "Atualizar horário da sessão", "id 99, horario = 21:45",
    ("21:45", 1), (sessao_teste["horario"], len(sessao_teste["ingressos"])))


testar(7, "Sessão lotada (parte do Luiz)", "id 12 (lotada) e id 1 (com vagas)",
    (True, False), (sessao_lotada(12), sessao_lotada(1)))


sessoes.remove(sessao_teste)       
print("\nRESULTADO FINAL:", total_ok, "de", total_testes, "testes passaram.")
print("Sessões após os testes:", len(sessoes))



def menu():
    while True:
        print("\n===== MÓDULO DE SESSÕES E ASSENTOS =====")
        print("1 - Cadastrar sessão")
        print("2 - Buscar sessão por ID")
        print("3 - Atualizar sessão")
        print("4 - Mapa de assentos")
        print("5 - Verificar se a sessão está lotada")
        print("0 - Sair")
        opcao = input("Escolha: ").strip()

        if opcao == "1":
            filme = ler_texto("Filme: ")
            sala = ler_texto("Sala: ")
            horario = ler_texto("Horário (ex.: 19:30): ")
            capacidade = ler_inteiro("Capacidade (1 a 100): ")
            ok, msg = cadastrar_sessao(proximo_id(sessoes), filme, sala, horario, capacidade)
            print(msg)
        elif opcao == "2":
            sessao = buscar_sessao(ler_inteiro("ID da sessão: "))
            if sessao is None:
                print("Sessão não encontrada.")
            else:
                print("Sessão:", sessao["id_sessao"], "|", sessao["filme"], "|", sessao["horario"],
                      "|", sessao["sala"], "| assentos livres:", len(assentos_livres(sessao)))
        elif opcao == "3":
            id_s = ler_inteiro("ID da sessão: ")
            campo = ler_texto("Campo (filme/horario/sala): ").lower()
            valor = ler_texto("Novo valor: ")
            print(atualizar_sessao(id_s, campo, valor)[1])
        elif opcao == "4":
            print(listar_assentos(ler_inteiro("ID da sessão: "))[1])
        elif opcao == "5":
            id_s = ler_inteiro("ID da sessão: ")
            if buscar_sessao(id_s) is None:
                print("Sessão não encontrada.")
            elif sessao_lotada(id_s):
                print("Sessão LOTADA.")
            else:
                print("Sessão com assentos livres.")
        elif opcao == "0":
            print("Encerrando o sistema. Até logo!")
            break
        else:
            print("Opção inválida. Escolha de 0 a 5.")

menu()

: 